In [32]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [33]:
df = pd.read_csv(r"C:\Users\saipr\OneDrive\Documents\Desktop\Music Recommendation System\Bollywood_Songs_With_Album_Genre.csv")

In [34]:
df.head()

,song_name,artist,release,lyrics,thumbnail,album,genre
0,Ek Haseena Thi Ek Deewana Tha,Yasser Desai,2017,Ek haseena thi ek deewana tha Kya kahun tumse ...,https://i.ytimg.com/vi/6obiArkHwAk/hqdefault.jpg,"Hue Bechain (From ""Ek Haseena Thi Ek Deewana T...","bollywood, hindi pop"
1,Hue Bechain,"Palak Muchhal, Yasser Desai",2017,Hue bechain pehli bar hmny raaz ye jana Mohabb...,https://i.ytimg.com/vi/6obiArkHwAk/hqdefault.jpg,"Hue Bechain (From ""Ek Haseena Thi Ek Deewana T...","bollywood, hindi pop"
2,Hanste Hanste,"Palak Muchhal, Yasser Desai",2017,Hanste hanste ro diye tum Kis mushkil mein kho...,https://i.ytimg.com/vi/rVKX19fNV0A/hqdefault.jpg,"Hanste Hanste (From ""Ek Haseena Thi Ek Deewana...","bollywood, hindi pop"
3,Nain,"Palak Muchhal, Yasser Desai",2017,Kuchh sawaal pyar ke lab pe hai ruke ruke Jaad...,https://i.ytimg.com/vi/ZHbbcy7u9Rk/hqdefault.jpg,"Nain (From ""Ek Haseena Thi Ek Deewana Tha"")","bollywood, hindi pop"
4,Aankhon Mein Aansoon,"Palak Muchhal, Yasser Desai",2017,Aankhon mein aansoo leke hoton se muskuraye Aa...,https://i.ytimg.com/vi/izP81UySf0Y/hqdefault.jpg,Unknown,Unknown


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   song_name  974 non-null    object
 1   artist     974 non-null    object
 2   release    974 non-null    int64 
 3   lyrics     974 non-null    object
 4   thumbnail  974 non-null    object
 5   album      974 non-null    object
 6   genre      974 non-null    object
dtypes: int64(1), object(6)
memory usage: 53.4+ KB


In [36]:
df.isna().sum()

song_name    0
artist       0
release      0
lyrics       0
thumbnail    0
album        0
genre        0
dtype: int64

In [37]:
df.duplicated().sum()

np.int64(0)

In [38]:
df.shape

(974, 7)

In [39]:
import re

def clean_text(text):
    text = str(text).lower()                        # lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)     # remove special characters
    text = re.sub(r'\s+', ' ', text).strip()        # remove extra spaces
    return text

df['song_name_clean'] = df['song_name'].apply(clean_text)
df['artist_clean'] = df['artist'].apply(clean_text)
df['album_clean'] = df['album'].apply(clean_text)
df['genre_clean'] = df['genre'].apply(clean_text)
df['lyrics_clean'] = df['lyrics'].apply(clean_text)

In [40]:
df['combined'] = (
    df['song_name'].astype(str) + " " +
    df['artist'].astype(str) + " " + 
    df['album'].astype(str) + " " +
    df['genre'].astype(str) + " " +
    df['lyrics'].astype(str)
)

In [41]:
df['combined']

0      Ek Haseena Thi Ek Deewana Tha Yasser Desai Hue...
1      Hue Bechain Palak Muchhal, Yasser Desai Hue Be...
2      Hanste Hanste Palak Muchhal, Yasser Desai Hans...
3      Nain Palak Muchhal, Yasser Desai Nain (From "E...
4      Aankhon Mein Aansoon Palak Muchhal, Yasser Des...
                             ...                        
969    Tauba Tumhare Ishare Abhijeet, Alka Yagnik Cha...
970    Sooraj Dooba Hain Arijit Singh, Aditi Singh Sh...
971    Manike (From "Thank God") Yohani, Jubin Nautiy...
972    Rangisari Kanishk Seth, Kavita Seth Rangisari ...
973    Chale Jaana Phir (Humko Tere Bina) Denny, Rahu...
Name: combined, Length: 974, dtype: object

In [42]:
tfidf = TfidfVectorizer(stop_words='english')
vectors = tfidf.fit_transform(df['combined'])

In [43]:
similarity = cosine_similarity(vectors)

In [44]:
def recommend(song):
    song = song.lower()

    try:
        index = df[df['song_name'].str.lower() ==song.index[0]]
    except:
        return[]
    distance = similarity[index]
    songs_list=sorted(list(enumerate(display)),
                      reverse=True,key=lambda x:x[1])[1:6]
    recommendation=[]
    for i in songs_list:
        recommendation.append((df.iloc[i][0]["song_name"]))
    return recommendation

In [45]:
pickle.dump(df,open("songs_df.pkl","wb"))
pickle.dump(similarity,open("similarity_df.pkl","wb"))
pickle.dump(tfidf,open("tfidf_df.pkl","wb"))

In [46]:
print(df.columns.tolist())

['song_name', 'artist', 'release', 'lyrics', 'thumbnail', 'album', 'genre', 'song_name_clean', 'artist_clean', 'album_clean', 'genre_clean', 'lyrics_clean', 'combined']
